In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

In [0]:
dbutils.widgets.text("secret_scope", "scope-trezio2005")
secret_scope = dbutils.widgets.get("secret_scope")

dbutils.widgets.text("evh_name", "trezio2005_evh")
evh_name = dbutils.widgets.get("evh_name")

dbutils.widgets.text("evh_space_name", "evhpl24databricks")
evh_space_name = dbutils.widgets.get("evh_space_name")

conn_string = dbutils.secrets.get(scope=secret_scope, key="trezio2005-evh-cs")

In [0]:
wikipedia_data_schema = StructType([
    StructField("user", StringType(), True),
    StructField("title", StringType(), True),
    StructField("type", StringType(), True),
    StructField("server_url", StringType(), True),
    StructField("timestamp", StringType(), True)]
)

df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", f"{evh_space_name}.servicebus.windows.net:9093")
      .option("subscribe", evh_name)
      .option("kafka.sasl.mechanism", "PLAIN")
      .option("kafka.security.protocol", "SASL_SSL")
      .option("kafka.sasl.jaas.config", f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="$ConnectionString" password="{conn_string}";')
      .option("startingOffsets", "earliest")
      .load()
)

processed_df = (df
    .withColumn("json_string", col("value").cast("string")) 
    .withColumn("parsed_data", from_json(col("json_string"), wikipedia_data_schema))
    .select("parsed_data.*") 
    .withColumn("ingestion_timestamp", current_timestamp()) 
)

In [0]:
dbutils.widgets.text("catalog_name", "dbr_dev")
dbutils.widgets.text("container", "trezio2005")
dbutils.widgets.text("storage_account", "dlspl21databricks")

catalog_name = dbutils.widgets.get("catalog_name")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

target_table = f"{catalog_name}.trezio2005_bronze.wikipedia_streaming_data"
checkpoint_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/checkpoints/wikipedia_stream"

In [0]:
(processed_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)